# Instruction fine-tuning para clasificación multiclase de textos

Dataset real: `SetFit/amazon_reviews_multi_es`  
Modelo instructivo: `google/flan-t5-base`

Transformamos la tarea en un problema de instrucción -> respuesta.

## 1. Instalación

In [ ]:
# !pip install -q transformers datasets evaluate accelerate sentencepiece
!pip install -q evaluate

## 2. Imports

In [ ]:
import numpy as np
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer
)

## 3. Cargar dataset

In [ ]:
dataset = load_dataset("SetFit/amazon_reviews_multi_es")
dataset


Cuando estás desarrollando es una buena idea que que selecciones una pequeña muestra de cada split y compruebes que tu notebook no tiene errores. Una vez  comprobado, ya sí podrías probar con todo el dataset (implicará mucho más tiempo). 

In [ ]:
train_ds = dataset["train"].shuffle(seed=42).select(range(3000))
val_ds = dataset["validation"].shuffle(seed=42).select(range(500))
test_ds = dataset["test"].shuffle(seed=42).select(range(500))

## 4. Formato instrucción-respuesta

In [ ]:
id2label_text = {
    0: "1 estrella",
    1: "2 estrellas",
    2: "3 estrellas",
    3: "4 estrellas",
    4: "5 estrellas",
}

def to_instruction(example):
    instruction = (
        "Clasifica la siguiente reseña de producto en una de estas categorías: "
        "1 estrella, 2 estrellas, 3 estrellas, 4 estrellas, 5 estrellas. "
        "Responde solo con la etiqueta."
    )
    return {
        "input_text": f"{instruction}\n\nReseña: {example['text']}",
        "target_text": id2label_text[example["label"]],
    }

train_inst = train_ds.map(to_instruction)
val_inst = val_ds.map(to_instruction)
test_inst = test_ds.map(to_instruction)

train_inst[0]

## 5. Tokenización

In [ ]:
model_checkpoint = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=256, truncation=True)
    labels = tokenizer(text_target=examples["target_text"], max_length=8, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_inst.map(preprocess_function, batched=True, remove_columns=train_inst.column_names)
tokenized_val = val_inst.map(preprocess_function, batched=True, remove_columns=val_inst.column_names)
tokenized_test = val_inst.map(preprocess_function, batched=True, remove_columns=test_inst.column_names)


## 6. Modelo

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

## 7. Métricas

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

label_text_to_id = {
    "1 estrella": 0,
    "2 estrellas": 1,
    "3 estrellas": 2,
    "4 estrellas": 3,
    "5 estrellas": 4,
}

def normalize_label(text):
    text = text.strip().lower()
    for label_text, label_id in label_text_to_id.items():
        if label_text in text:
            return label_id
    for n in ["1", "2", "3", "4", "5"]:
        if n in text:
            return int(n) - 1
    return None

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_ids = [normalize_label(x) for x in decoded_preds]
    label_ids = [normalize_label(x) for x in decoded_labels]
    valid = [(p, y) for p, y in zip(pred_ids, label_ids) if p is not None and y is not None]
    if not valid:
        return {"accuracy": 0.0, "macro_f1": 0.0}

    pred_ids = [p for p, _ in valid]
    label_ids = [y for _, y in valid]

    return {
        "accuracy": accuracy.compute(predictions=pred_ids, references=label_ids)["accuracy"],
        "macro_f1": f1.compute(predictions=pred_ids, references=label_ids, average="macro")["f1"],
    }

## 8. Entrenamiento

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./ift_amazon_reviews_es",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 9. Lanzar entrenamiento

In [ ]:
trainer.train()

## 10. Evaluación sobre el conjunto de validación

In [ ]:
pred_output = trainer.predict(tokenized_val)
pred_output.metrics

# 11. Evaluación sobre el conjunto test

Si el conjunto test ya incluye las labels, puedes evaluar el modelo sobre dicho conjunto.

In [ ]:
pred_output = trainer.predict(tokenized_test)
pred_output.metrics

## 12. Inferencia 

El siguiente código se podría aplicar sobre los textos del conjunto test, o bien sobre una pequeña muestra:

In [ ]:
import torch

examples = [
    "Muy mala compra, el producto se rompió enseguida.",
    "No está mal, pero esperaba algo mejor.",
    "Excelente, funciona perfecto y llegó muy rápido.",
    "Horrible!!!.",
]

prompts = [
    "Clasifica la siguiente reseña de producto en una de estas categorías: "
    "1 estrella, 2 estrellas, 3 estrellas, 4 estrellas, 5 estrellas. "
    "Responde solo con la etiqueta.\n\nReseña: " + x
    for x in examples
]

device = model.device
inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=8)

decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

for text, pred in zip(examples, decoded):
    print(text)
    print("Predicción:", pred)
    print("-" * 60)

El modelo parece que sólo asina la clase 3 estrellas (clase 2). Vamos a comprobarlo:


In [ ]:
pred_output = trainer.predict(tokenized_test)
decoded_preds = tokenizer.batch_decode(pred_output.predictions, skip_special_tokens=True)

normalized_preds = [normalize_label(x) for x in decoded_preds]

from collections import Counter
print(Counter(normalized_preds))

Eso suele pasar por una de estas razones:

- Datos de entrenamiento insuficientes (entrena con más dataset y comprueba si se mantiene el problema). Si estás usando un subconjunto pequeño y pocas épocas, el modelo puede no aprender bien la frontera entre clases y quedarse con una salida dominante.

- El modelo aprende la clase más "segura". En una tarea de 5 estrellas, 3 estrellas suele ser una salida fácil porque es la clase intermedia, una opción neutra, y cuando el modelo no está seguro, tiende a elegir una respuesta conservadora.

- Desbalance de clases: Si 3 estrellas (clase 2) aparece mucho, el modelo puede sesgarse hacia ahí. En la siguiente celda, podemos ver que en efecto la clase 2 (3 estrellas) es la más frecuente.

- Es posible que el prompt no esté bien definido. En esos casos, el modelo puede aprender una respuesta frecuente y estable, como “3 estrellas”.

En la siguiente celda, podemos ver que no es un problema de desbalance de clases:

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Contar clases
train_counts = Counter(train_ds["label"])

# Ordenar por clase
labels = sorted(train_counts.keys())
values = [train_counts[l] for l in labels]

# Plot
plt.figure()
plt.bar(labels, values)
plt.xlabel("Clase (label)")
plt.ylabel("Frecuencia")
plt.title("Distribución de clases (train)")
plt.xticks(labels)
plt.show()

In [ ]:
print(Counter(test_ds["label"]))

Otro posible error bastante frecuente es cómo parseamos la salida que ha generado el modelo. El modelo es un modelo generativo, así que no siempre responde como si fuera un problema de clasificación cerrada, sino que puede generar distintos textos como salida. 

Por ejemplo, si genera cosas como:
- "3"
- "3 estrellas."
- "La reseña parece de 3 estrellas"

tu parser puede estar convirtiendo demasiadas salidas a la misma clase.

Es una práctica necesaria que siempre mires las predicciones brutas del modelo:

In [ ]:
sample_preds = trainer.predict(tokenized_test)
decoded_preds = tokenizer.batch_decode(sample_preds.predictions, skip_special_tokens=True)

for i in range(20):
    print("Predicción bruta:", repr(decoded_preds[i]))

En resumen, es posible que: 
- estemos entrenando con pocos datos,
- con pocas épocas,
- y tarea demasiado fina (5 clases).

Posibles soluciones:
- entrenar con una muestra mayor
- pasar de 2 a 3 epoch
- Siempre debes comprobar que la normalización de la salida funciona correctamente (observa la predicción bruta del modelo y comprueba que la función de normalización es adecuada).
- Probar con otro modelo generativo. Por ejemplo, google/flan-t5-large.